In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Advanced Short-Answer Scoring with TF–IDF + Thai BERT + XGBoost + Upgrades
Author: Your Name

Description:
  This script extends the original approach with:
    1) Thai tokenization + stopword removal (PyThaiNLP) for better TF–IDF.
    2) Mean-pooling for BERT instead of just [CLS] token.
    3) Truncated SVD on TF–IDF vectors to reduce dimensionality.
    4) Simple ensemble: Average predictions from XGBoost & RandomForest.

Dependencies (install with pip):
    scikit-learn, xgboost, transformers, pythainlp, tqdm

Usage:
    python advanced_shortanswer.py
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# For Thai tokenization and stopwords
try:
    from pythainlp import word_tokenize
    from pythainlp.corpus.common import thai_stopwords
    THAI_STOPWORDS = set(thai_stopwords())
    USE_THAI_NLP = True
except ImportError:
    print("pythainlp not installed or no stopwords, continuing without advanced Thai tokenization.")
    USE_THAI_NLP = False


###############################################################################
# 1) Reading Data
###############################################################################
def read_data():
    """
    Reads train.csv, test.csv, and sample_submission.csv from current directory.
    Expects:
      train.csv => [ID, set, question, answer, score]
      test.csv  => [ID, set, question, answer]
      sample_submission.csv => [ID, score]
    """
    train_path = os.path.join(REPO_PATH, "data", "train.csv")
    test_path = os.path.join(REPO_PATH, "data", "test.csv")
    sample_sub_path = os.path.join(REPO_PATH, "data", "sample_submission.csv")

    for p in [train_path, test_path, sample_sub_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")

    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    sub_df   = pd.read_csv(sample_sub_path)
    return train_df, test_df, sub_df


###############################################################################
# 2) Text Combination + Thai Tokenization
###############################################################################
def combine_text(question, answer):
    """
    Combine question + answer into one string.
    Upgraded to tokenize Thai text and remove stopwords (if pythainlp is available).
    """
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    combined = q + " " + a

    if USE_THAI_NLP:
        # Thai tokenization
        tokens = word_tokenize(combined, keep_whitespace=False)
        # Remove stopwords
        tokens = [t for t in tokens if t not in THAI_STOPWORDS]
        return " ".join(tokens)
    else:
        # fallback: just return the raw text
        return combined


###############################################################################
# 3) TF–IDF Vectorizer
###############################################################################
def build_tfidf_vectorizer():
    """
    Create a TfidfVectorizer. We use unigrams + bigrams, ignore min_df < 2, up to 10k features.
    """
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=10000
    )
    return vectorizer


###############################################################################
# 4) Thai BERT Embedder (Mean Pooling)
###############################################################################
class ThaiBERTEmbedder:
    """
    Use a pretrained Thai BERT (or multilingual) model from Hugging Face,
    applying MEAN pooling rather than the [CLS] token.
    """
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading BERT tokenizer/model for: {model_name} on device: {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

    def mean_pooling(self, last_hidden_state, attention_mask):
        """
        Mean pool the token embeddings, ignoring padding.
        """
        # last_hidden_state: [batch_size, seq_len, hidden_dim]
        # attention_mask: [batch_size, seq_len]
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = torch.sum(last_hidden_state * mask, 1)
        counts = torch.clamp(mask.sum(1), min=1e-9)
        return summed / counts

    def encode(self, text_list, batch_size=16, max_length=128):
        """
        Convert text_list into BERT embeddings using mean pooling.
        Returns shape = [len(text_list), hidden_dim].
        """
        all_embs = []
        for i in range(0, len(text_list), batch_size):
            batch_text = text_list[i: i + batch_size]
            inputs = self.tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
            last_hidden = outputs.last_hidden_state  # [B, seq_len, hidden_dim]
            pooled = self.mean_pooling(last_hidden, inputs["attention_mask"])
            all_embs.append(pooled.cpu().numpy())

        return np.concatenate(all_embs, axis=0)


###############################################################################
# 5) Preprocess: TF–IDF + BERT + Dim Reduction
###############################################################################
def preprocess_data(
    train_df,
    test_df,
    tfidf_vectorizer,
    bert_embedder,
    svd_components=300
):
    """
    1) Combine Q + A (with Thai tokenization).
    2) Fit TF–IDF on train, transform test => X_tfidf_train, X_tfidf_test
    3) BERT embeddings => X_bert_train, X_bert_test
    4) (Optional) TruncatedSVD on TF–IDF to reduce dimensionality
    5) Stack [tfidf_svd, bert] => final features
    """
    # Combine Q + A
    train_df["text"] = train_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)

    # TF–IDF
    print("Fitting TF–IDF on train data...")
    X_tfidf_train = tfidf_vectorizer.fit_transform(train_df["text"].tolist())
    print("Transforming test data with TF–IDF...")
    X_tfidf_test  = tfidf_vectorizer.transform(test_df["text"].tolist())

    # BERT
    print("Generating BERT embeddings for train...")
    X_bert_train = bert_embedder.encode(train_df["text"].tolist())
    print("Generating BERT embeddings for test...")
    X_bert_test  = bert_embedder.encode(test_df["text"].tolist())

    # Reduce TF–IDF dimension with truncated SVD
    print(f"Performing TruncatedSVD to {svd_components} components on TF–IDF.")
    svd = TruncatedSVD(n_components=svd_components, random_state=42)
    X_tfidf_train_svd = svd.fit_transform(X_tfidf_train)
    X_tfidf_test_svd  = svd.transform(X_tfidf_test)

    # Stack horizontally: [tfidf_svd, bert]
    X_train = np.hstack([X_tfidf_train_svd, X_bert_train])
    X_test  = np.hstack([X_tfidf_test_svd,  X_bert_test])

    y_train = train_df["score"].values
    return X_train, y_train, X_test


###############################################################################
# 6) Cross-Validation on XGBoost
###############################################################################
def run_cross_val(X, y, n_splits=5):
    """
    Simple KFold cross-validation to estimate MSE for XGBoost only.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    mses = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            tree_method="auto",  # 'gpu_hist' if GPU is available
            early_stopping_rounds=20,
        )
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  verbose=False)

        y_pred_val = model.predict(X_val)
        fold_mse = mean_squared_error(y_val, y_pred_val)
        mses.append(fold_mse)
    return mses


###############################################################################
# 7) Final Training & Simple Ensemble
###############################################################################
def train_xgb_model(X, y):
    """
    Train final XGBoost on full data.
    """
    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="auto",
    )
    model.fit(X, y, verbose=False)
    return model


def train_rf_model(X, y):
    """
    Train a simple RandomForestRegressor on full data (default params).
    """
    rf = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X, y)
    return rf


def predict_and_save(xgb_model, rf_model, X_test, sub_df, output_name="submission.csv"):
    """
    Predict with both XGB + RF, average their predictions, and save CSV.
    """
    pred_xgb = xgb_model.predict(X_test)
    pred_rf  = rf_model.predict(X_test)

    # Simple ensemble by averaging
    predictions = (pred_xgb + pred_rf) / 2.0

    # If your score range is known [0,5], you can clip:
    # predictions = np.clip(predictions, 0, 5)

    sub_df["score"] = predictions
    sub_df.to_csv(output_name, index=False)
    print(f"Submission saved => {output_name}")


###############################################################################
# Main Script
###############################################################################
def main():
    print("=== (1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== (2) Build TF–IDF Vectorizer ===")
    tfidf_vectorizer = build_tfidf_vectorizer()

    print("=== (3) Init Thai BERT embedder (mean pooling) ===")
    bert_embedder = ThaiBERTEmbedder(
        model_name="airesearch/wangchanberta-base-att-spm-uncased",
        device=None  # auto GPU
    )

    print("=== (4) Preprocessing + Dim Reduction ===")
    X_train, y_train, X_test = preprocess_data(
        train_df,
        test_df,
        tfidf_vectorizer,
        bert_embedder,
        svd_components=300
    )

    print(f"Train shape = {X_train.shape}, Test shape = {X_test.shape}")

    print("=== (5) Cross-validation (XGB only) ===")
    cv_mses = run_cross_val(X_train, y_train, n_splits=5)
    print("Fold MSEs:", cv_mses)
    print("Mean CV MSE:", np.mean(cv_mses))

    print("=== (6) Train final XGB + RF on all data ===")
    final_xgb = train_xgb_model(X_train, y_train)
    final_rf  = train_rf_model(X_train, y_train)

    print("=== (7) Predict & Save (ensemble) ===")
    predict_and_save(final_xgb, final_rf, X_test, sub_df, output_name="submission.csv")

    print("All done! Check submission.csv for predictions.")


if __name__ == "__main__":
    main()


=== (1) Reading data ===
=== (2) Build TF–IDF Vectorizer ===
=== (3) Init Thai BERT embedder (mean pooling) ===
Loading BERT tokenizer/model for: airesearch/wangchanberta-base-att-spm-uncased on device: cpu
=== (4) Preprocessing + Dim Reduction ===
Fitting TF–IDF on train data...
Transforming test data with TF–IDF...
Generating BERT embeddings for train...
Generating BERT embeddings for test...
Performing TruncatedSVD to 300 components on TF–IDF.
Train shape = (362, 1068), Test shape = (90, 1068)
=== (5) Cross-validation (XGB only) ===
Fold MSEs: [np.float64(2.2755401532388047), np.float64(2.3417003821416382), np.float64(2.406601022730652), np.float64(2.0651984401344787), np.float64(2.4909251505333017)]
Mean CV MSE: 2.315993029755775
=== (6) Train final XGB + RF on all data ===


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


=== (7) Predict & Save (ensemble) ===
Submission saved => submission.csv
All done! Check submission.csv for predictions.
